# 4.3 · Testing and debugging: Inspector and logs

**测试与调试：Inspector 与日志**

> **After this lesson you can:** Debug an MCP server with the Inspector (web, CLI, TUI, mcpdo) in both protocol eras, read host and server logs, fix the classic startup failures, and lock behaviour in with automated contract and golden tests.

Official reading: [MCP Inspector](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector) · [Inspector CLI client](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector/cli) · [Inspector protocol eras](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector/protocol-eras) · [Debugging](https://modelcontextprotocol.io/docs/2026-07-28/tools/debugging)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. The Inspector commands you will actually type
The Inspector is a Node package fetched by `npx`, so this offline notebook does not run it. Keep these commands next to your terminal (replace `server.py` with your entry point):

```bash
# explore in the browser (prints a URL with a per-launch token)
npx @modelcontextprotocol/inspector uv run server.py

# one-shot CLI checks, in both protocol eras
npx @modelcontextprotocol/inspector --cli uv run server.py --protocol-era modern --method tools/list
npx @modelcontextprotocol/inspector --cli uv run server.py --protocol-era legacy --method tools/list
npx @modelcontextprotocol/inspector --cli uv run server.py --method tools/call \
    --tool-name notes_search --tool-arg query=mcp --format json | jq .result

# pass env and cwd to a stdio server; CI gate on schema portability
npx @modelcontextprotocol/inspector --cli -e NOTES_TOKEN=dev --cwd ~/notes uv run server.py --method tools/list --strict

# terminal UI, and a persistent connection
npx @modelcontextprotocol/inspector --tui uv run server.py
npx -p @modelcontextprotocol/inspector mcpdo connect @notes uv run server.py
```

Everything below rebuilds the core of those workflows in Python, so you can see what the tools do and reuse the ideas in your own tests.

### 2. A server under test
A small notes server with tools, a resource, a template and a prompt. It resolves its data file **relative to its own source file** and logs to **stderr**: two habits this lesson explains.

In [ ]:
import json
import os
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
workdir = Path(tempfile.mkdtemp(prefix="inspect-"))
server_path = workdir / "notes_server.py"
(workdir / "notes.json").write_text(json.dumps({
    "mcp": "MCP is USB-C for AI applications.",
    "lsp": "The Language Server Protocol inspired MCP.",
    "eras": "The Inspector supports legacy, auto and modern protocol eras.",
}))
print("workdir:", workdir)

In [ ]:
%%writefile {server_path}
import json
import logging
import sys
from pathlib import Path
from typing import Annotated, Literal

from pydantic import Field

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from mcp_types import ToolAnnotations

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("notes")
NOTES: dict[str, str] = json.loads((Path(__file__).parent / "notes.json").read_text())
log.info("loaded %d notes", len(NOTES))

mcp = MCPServer("notes", version="1.2.0")
RO = ToolAnnotations(readOnlyHint=True, openWorldHint=False)


@mcp.tool(annotations=RO)
def notes_search(query: Annotated[str, Field(description="Case-insensitive text to look for")],
                 match: Annotated[Literal["any", "all"], Field(description="Match any or all words")] = "any") -> list[str]:
    """Return the ids of notes whose text matches the query. Read one with notes_get."""
    words = query.lower().split()
    test = any if match == "any" else all
    return sorted(nid for nid, text in NOTES.items() if test(w in text.lower() for w in words))


@mcp.tool(annotations=RO)
def notes_get(note_id: Annotated[str, Field(description="A note id from notes_search")]) -> str:
    """Return the full text of one note."""
    if note_id not in NOTES:
        raise ToolError(f"Unknown note {note_id!r}. Call notes_search to list valid ids.")
    return NOTES[note_id]


@mcp.resource("notes://index")
def index() -> str:
    """All note ids, one per line."""
    return "\n".join(sorted(NOTES))


@mcp.resource("notes://{note_id}")
def note(note_id: str) -> str:
    """One note as a resource."""
    return NOTES[note_id]


@mcp.prompt()
def summarize(note_id: str) -> str:
    """Ask the model to summarise a note."""
    return f"Summarise note {note_id} in one sentence: {NOTES.get(note_id, '')}"


if __name__ == "__main__":
    mcp.run(transport="stdio")

### 3. A mini Inspector
The Inspector CLI connects, runs one method and exits with a meaningful code. Our `inspect_server` does the "list everything, then call every tool" pass. It builds **sample arguments from each `inputSchema`** (first enum value, default, or a type-based placeholder) and returns an Inspector-style exit code: `0` when all calls succeed, `5` when any tool returns `isError`.

In [ ]:
import time

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client


def sample_args(schema: dict, overrides: dict | None = None) -> dict:
    samples = {"string": "mcp", "integer": 1, "number": 1.0, "boolean": True, "array": [], "object": {}}
    args = {}
    for name, prop in schema.get("properties", {}).items():
        if name not in schema.get("required", []):
            continue                                    # like a form left at its defaults
        if "enum" in prop:
            args[name] = prop["enum"][0]
        else:
            args[name] = prop.get("default", samples.get(prop.get("type"), "mcp"))
    return args | (overrides or {})


async def inspect_server(target, overrides: dict | None = None) -> int:
    exit_code = 0
    async with Client(target) as c:
        caps = c.server_capabilities.model_dump(exclude_none=True)
        print(f"server: {c.server_info.name} {c.server_info.version} | protocol {c.protocol_version}")
        print("capabilities:", sorted(caps))
        tools = (await c.list_tools()).tools
        print("resources:", [str(r.uri) for r in (await c.list_resources()).resources],
              "| templates:", [t.uri_template for t in (await c.list_resource_templates()).resource_templates],
              "| prompts:", [p.name for p in (await c.list_prompts()).prompts])
        for tool in tools:
            args = sample_args(tool.input_schema, (overrides or {}).get(tool.name))
            started = time.perf_counter()
            r = await c.call_tool(tool.name, args)
            ms = (time.perf_counter() - started) * 1000
            status = "ERROR" if r.is_error else "ok"
            exit_code = 5 if r.is_error else exit_code
            hints = tool.annotations.model_dump(by_alias=True, exclude_none=True) if tool.annotations else {}
            print(f"  {status:5} {tool.name}({json.dumps(args)}) {ms:5.1f} ms  hints={hints}")
            shown = r.structured_content if r.structured_content is not None else r.content[0].text
            print(f"        -> {json.dumps(shown)[:90]}")
    return exit_code


params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
with open(workdir / "notes.stderr.log", "w") as errlog:
    code = await inspect_server(stdio_client(params, errlog=errlog))
print("exit code:", code)

`notes_get("mcp")` happened to succeed because our placeholder string is a real id. Try `overrides={"notes_get": {"note_id": "nope"}}` and the exit code becomes `5`, just like `mcp-inspector --cli … --method tools/call` when a tool returns `isError: true`.

### 4. A wire recorder: the Protocol tab, in 30 lines
The Inspector's Protocol tab is a transcript of JSON-RPC messages. We get the same by putting a tiny **stdio proxy** between client and server. It forwards every line unchanged and appends it to a JSONL transcript, flagging anything on stdout that is not JSON. Because it is just another command, it works with any stdio client, including Claude Desktop (`"command": "python", "args": ["proxy.py", "log.jsonl", "python", "server.py"]`).

In [ ]:
proxy_path = workdir / "wire_proxy.py"
proxy_path.write_text('''
import json, subprocess, sys, threading, time
log_path, *cmd = sys.argv[1:]
child = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE)
lock = threading.Lock()

def record(direction, raw):
    text = raw.decode("utf-8", "replace").rstrip("\\n")
    try:
        entry = {"t": time.time(), "dir": direction, "msg": json.loads(text)}
    except json.JSONDecodeError:
        entry = {"t": time.time(), "dir": direction, "invalid": text}
    with lock, open(log_path, "a") as f:
        f.write(json.dumps(entry) + "\\n")

def client_to_server():
    for line in sys.stdin.buffer:
        record("->", line)
        child.stdin.write(line)
        child.stdin.flush()
    child.stdin.close()

threading.Thread(target=client_to_server, daemon=True).start()
for line in child.stdout:
    record("<-", line)
    sys.stdout.buffer.write(line)
    sys.stdout.buffer.flush()
sys.exit(child.wait())
''')


def show_transcript(path: Path, width: int = 150) -> list[dict]:
    entries = [json.loads(l) for l in path.read_text().splitlines()]
    for e in entries:
        if "invalid" in e:
            print(f"{e['dir']} !! NOT JSON-RPC: {e['invalid']!r}")
            continue
        m = e["msg"]
        kind = m.get("method") or ("result" if "result" in m else "error")
        body = m.get("params") or m.get("result") or m.get("error")
        print(f"{e['dir']} id={m.get('id', '-')!s:4} {kind:22} {json.dumps(body)[:width]}")
    return entries


transcript = workdir / "wire.jsonl"
recorded = StdioServerParameters(command=sys.executable,
                                 args=[str(proxy_path), str(transcript), sys.executable, str(server_path)])
with open(workdir / "notes.stderr.log", "w") as errlog:
    async with Client(stdio_client(recorded, errlog=errlog)) as c:
        await c.call_tool("notes_search", {"query": "protocol eras", "match": "all"})
        await c.call_tool("notes_get", {"note_id": "nope"})
entries = show_transcript(transcript)

Read the transcript top to bottom: the client's `server/discover` probe, the 2026-07-28 `_meta` envelope on every request, a `tools/list` the SDK client sent after the first call (to fetch the tool's `outputSchema` and validate `structuredContent`), and the `isError: true` result for the unknown note, which is a *result*, not a JSON-RPC error.

In [ ]:
first_call = next(e["msg"] for e in entries if e["dir"] == "->" and e["msg"].get("method") == "tools/call")
print("one complete request, exactly as written to the server's stdin:")
print(json.dumps(first_call, indent=2))

### 5. Protocol compatibility checks by hand
The debugging guide's connection checklist ends with "verify protocol compatibility" and "check the per-request `_meta` fields". Here are the three answers you should recognise: a good `server/discover`, a missing `_meta` (`-32602`), and an unsupported version (`-32022` with `data.supported`).

In [ ]:
import subprocess

proc = subprocess.Popen([sys.executable, str(server_path)], text=True,
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0"}}


def rpc(id_, method, params):
    proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}) + "\n")
    proc.stdin.flush()
    return json.loads(proc.stdout.readline())


ok = rpc(1, "server/discover", {"_meta": META})
print("server/discover ->", {k: ok["result"][k] for k in ("supportedVersions", "capabilities")})
print("missing _meta   ->", rpc(2, "tools/list", {})["error"])
old = dict(META, **{"io.modelcontextprotocol/protocolVersion": "2020-01-01"})
print("bad version     ->", rpc(3, "tools/list", {"_meta": old})["error"])
proc.stdin.close()
_ = proc.wait(timeout=10)

### 6. A test suite: behaviour, contract and golden wire tests
We write a real pytest-style test file next to the server. It uses in-memory `Client(server)` for speed and one stdio smoke test. The first run records the **golden files** (the tool contract and a wire-format result). Later runs compare against them. If `pytest` is installed it runs the file directly; otherwise a 15-line runner executes the same test functions, awaiting the async ones.

In [ ]:
tests_path = workdir / "test_notes_server.py"
tests_path.write_text('''
import importlib.util, json, os, sys
from pathlib import Path
import pytest
from mcp import Client, StdioServerParameters

HERE = Path(__file__).parent
GOLDEN = HERE / "golden"
spec = importlib.util.spec_from_file_location("notes_server", HERE / "notes_server.py")
notes_server = importlib.util.module_from_spec(spec)
spec.loader.exec_module(notes_server)
server = notes_server.mcp


def wire(model):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)                      # volatile/identity fields don't belong in goldens
    return data


def check_golden(name, value):
    path = GOLDEN / f"{name}.json"
    text = json.dumps(value, indent=2, sort_keys=True)
    if os.environ.get("UPDATE_GOLDEN") or not path.exists():
        GOLDEN.mkdir(exist_ok=True)
        path.write_text(text)
    if text != path.read_text():
        path.with_suffix(".actual.json").write_text(text)     # keep the new version for the diff
        raise AssertionError(f"{name} changed; review the diff, then re-run with UPDATE_GOLDEN=1")


@pytest.mark.anyio
async def test_search_finds_notes():
    async with Client(server) as c:
        r = await c.call_tool("notes_search", {"query": "protocol"})
    assert not r.is_error and r.structured_content == {"result": ["eras", "lsp"]}


@pytest.mark.anyio
async def test_unknown_note_is_actionable_error():
    async with Client(server) as c:
        r = await c.call_tool("notes_get", {"note_id": "nope"})
    assert r.is_error and "notes_search" in r.content[0].text


@pytest.mark.anyio
async def test_tool_contract():
    async with Client(server) as c:
        tools = [wire(t) for t in (await c.list_tools()).tools]
    check_golden("tools_contract", tools)


@pytest.mark.anyio
async def test_golden_wire_result():
    async with Client(server) as c:
        r = await c.call_tool("notes_search", {"query": "mcp"})
    check_golden("notes_search_mcp", wire(r))


@pytest.mark.anyio
async def test_stdio_smoke():
    params = StdioServerParameters(command=sys.executable, args=[str(HERE / "notes_server.py")], cwd="/")
    async with Client(params) as c:              # cwd="/" like a GUI host; catches path bugs
        assert {t.name for t in (await c.list_tools()).tools} == {"notes_search", "notes_get"}
''')

import difflib
import importlib.util
import inspect
import logging
import shutil

for name in ("notes", "mcp"):                       # keep server-side logs out of the test report
    logging.getLogger(name).setLevel(logging.WARNING)


HAVE_PYTEST = importlib.util.find_spec("pytest") is not None


async def run_tests(path: Path) -> bool:
    if HAVE_PYTEST:
        result = subprocess.run([sys.executable, "-m", "pytest", "-q", str(path)],
                                capture_output=True, text=True, cwd=path.parent)
        print(result.stdout[-1500:])
        return result.returncode == 0
    sys.modules.setdefault("pytest", type(sys)("pytest"))           # minimal stand-in for pytest.mark.anyio
    sys.modules["pytest"].mark = type("M", (), {"anyio": staticmethod(lambda f: f)})
    spec = importlib.util.spec_from_file_location("test_notes_server", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    passed = True
    for name, fn in inspect.getmembers(module, inspect.isfunction):
        if not name.startswith("test_"):
            continue
        try:
            await fn() if inspect.iscoroutinefunction(fn) else fn()
            print(f"PASS {name}")
        except Exception as exc:
            passed = False
            print(f"FAIL {name}: {type(exc).__name__}: {str(exc)[:300]}")
    return passed


shutil.rmtree(workdir / "golden", ignore_errors=True)
print("run 1 (records goldens):", await run_tests(tests_path))
print("goldens:", sorted(p.name for p in (workdir / "golden").iterdir()))
print("\nrun 2 (compares):", await run_tests(tests_path))

Now make an innocent-looking change, renaming the parameter `match` to `mode`, and run the suite again. The behaviour tests still pass, but the **contract test** catches a breaking change for every client and prompt that used `match`.

In [ ]:
original_source = server_path.read_text()
server_path.write_text(original_source.replace("match: Annotated", "mode: Annotated")
                       .replace('if match == "any"', 'if mode == "any"'))
print("after renaming a parameter:", await run_tests(tests_path))
golden = workdir / "golden" / "tools_contract.json"
print("".join(l for l in difflib.unified_diff(golden.read_text().splitlines(True),
                                               golden.with_suffix(".actual.json").read_text().splitlines(True),
                                               "golden", "actual", n=0) if not l.startswith("@@")))
_ = server_path.write_text(original_source)           # revert

### 7. Debug a broken server, step by step
Here is a server that "works on my machine": it runs fine from its own folder in a terminal. A host launches it differently, though: absolute script path, working directory `/`, and a minimal environment. We launch it the way a host would, read the stderr log the way you would read `mcp-server-NAME.log`, and fix one bug per step.

In [ ]:
broken_path = workdir / "broken_notes.py"
broken_path.write_text('''
import json, os
from mcp.server import MCPServer

NOTES = json.load(open("notes.json"))           # bug 1: relative to the *working directory*
TOKEN = os.environ["NOTES_TOKEN"]               # bug 2: env var the host does not pass
print(f"Loaded {len(NOTES)} notes", flush=True) # bug 3: stdout is the wire

mcp = MCPServer("broken-notes")

@mcp.tool()
def count_notes() -> int:
    """How many notes exist."""
    return len(NOTES)

if __name__ == "__main__":
    mcp.run()
''')


def leaf_error(exc: BaseException) -> BaseException:
    while isinstance(exc, BaseExceptionGroup):
        exc = exc.exceptions[0]
    return exc


async def launch_like_a_host(env: dict | None = None, record_to: Path | None = None) -> None:
    log_path = workdir / "mcp-server-broken-notes.log"
    command = [str(broken_path)]
    if record_to:
        command = [str(proxy_path), str(record_to), sys.executable, str(broken_path)]
    params = StdioServerParameters(command=sys.executable, args=command, cwd="/", env=env)
    try:
        with open(log_path, "w") as errlog:
            async with Client(stdio_client(params, errlog=errlog)) as c:
                r = await c.call_tool("count_notes", {})
                print("CONNECTED: count_notes ->", r.structured_content)
    except Exception as exc:
        print("CLIENT SAW:", repr(leaf_error(exc)))
        tail = log_path.read_text().strip().splitlines()[-1:]
        print("LOG TAIL  :", tail)


print("step 1: launch it like a host would")
await launch_like_a_host()

The client only sees `Connection closed`, which is all a host can tell you. The **log** has the cause: `notes.json` was looked up in `/`. Fix 1: resolve data relative to the source file.

In [ ]:
src = broken_path.read_text()
src = src.replace('json.load(open("notes.json"))',
                  'json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "notes.json")))')
broken_path.write_text(src)
print("step 2: after fixing the path")
await launch_like_a_host()

New failure, new log line: `KeyError: 'NOTES_TOKEN'`. Stdio servers inherit only a limited environment, so pass it explicitly, which is the equivalent of an `"env"` entry in `claude_desktop_config.json`.

In [ ]:
print("step 3: pass the environment explicitly, and record the wire")
wire_log = workdir / "broken-wire.jsonl"
await launch_like_a_host(env={"NOTES_TOKEN": "dev-token"}, record_to=wire_log)
_ = show_transcript(wire_log, width=90)

It connects, because the Python client tolerates the junk line, but the recorder exposes **bug 3**: a non-JSON line on stdout before the first response. A stricter host could drop the connection here. Fix 3: log to stderr. Then confirm the transcript is clean.

In [ ]:
broken_path.write_text(broken_path.read_text().replace(
    'print(f"Loaded {len(NOTES)} notes", flush=True)', 'print(f"Loaded {len(NOTES)} notes", file=sys.stderr)')
    .replace("import json, os", "import json, os, sys"))
wire_log.unlink()
print("step 4: after logging to stderr")
await launch_like_a_host(env={"NOTES_TOKEN": "dev-token"}, record_to=wire_log)
bad_lines = [e for e in map(json.loads, wire_log.read_text().splitlines()) if "invalid" in e]
print("non-JSON lines on the wire:", bad_lines)
print("server log:", (workdir / "mcp-server-broken-notes.log").read_text().strip())

Each bug became visible only by **reproducing the host's launch conditions** and **reading the right log**. Turn every fix into a regression test: section 6's `test_stdio_smoke` already launches with `cwd="/"`.

### Your turn
1. Add a third tool to `notes_server.py` (say `notes_count`) and re-run section 6. Which test fails, and what command (environment variable) accepts the new contract?
2. Extend `inspect_server` to read every static resource and render every prompt with sample arguments, reporting failures the way it reports tools.
3. Run `inspect_server` in-process with `Client(..., mode="legacy")` by loading the server module. Which protocol version does it report, and does anything else change?
4. Make the wire recorder also measure latency: pair each `->` request with its `<-` response by `id` and print the milliseconds per method.